In [ ]:
from dd_cleaner.notebook_utils import prepare_workspace, init_notebook_session

# 1. Standardize KMDS structure within the tests/ directory
# This ensures /home/rajiv/programming/dd_parser_cleaner/tests/scripts/ exists
prepare_workspace(working_dir="/home/rajiv/programming/dd_parser_cleaner/tests")

# 2. Initialize the session and load the raw dataset
# This adds the local scripts/ folder to your Python path automatically
coord, df = init_notebook_session(working_dir="/home/rajiv/programming/dd_parser_cleaner/tests")

print(f"✅ Migration session initialized for workspace: {coord.base_dir}")


✅ Migration session initialized for workspace: /home/rajiv/programming/dd_parser_cleaner/tests


In [4]:
import sys
import importlib
import pandas as pd
from scripts import domain_logic  # Path 2 standard: logic is in the scripts/ subfolder

# 1. Force a reload to pick up the new 'impute_categorical_missing' function
importlib.reload(domain_logic)

# 2. Dynamically find a categorical column that actually has missing values
# This avoids the "Bank" column hallucination
test_col = None
for col in df.columns:
    if df[col].dtype == 'object' and df[col].isna().any():
        test_col = col
        break

if not test_col:
    print("⚠️ No categorical columns with missing values found in this sample.")
    # Create a dummy missing value for verification if needed
    test_col = df.select_dtypes(include=['object']).columns[0]
    df.loc[0, test_col] = None
    print(f"Added a dummy null to '{test_col}' for verification purposes.")

print(f"🔍 Testing on column: '{test_col}'")
print(f"   Pre-cleaning null count: {df[test_col].isna().sum()}")

# 3. Apply the custom logic
# We call it through the 'domain_logic' module reference
df[test_col] = domain_logic.impute_categorical_missing(df, test_col)

# 4. Final verification
null_count = df[test_col].isna().sum()
print(f"   Post-cleaning null count: {null_count}")

if "MISSING" in df[test_col].values and null_count == 0:
    print(f"✅ Logic verified! '{test_col}' now uses 'MISSING' category.")
else:
    print("❌ Verification failed. Check if the function was saved to scripts/domain_logic.py")


⚠️ No categorical columns with missing values found in this sample.
Added a dummy null to 'asofdate' for verification purposes.
🔍 Testing on column: 'asofdate'
   Pre-cleaning null count: 1


/tmp/ipykernel_60381/834638519.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  test_col = df.select_dtypes(include=['object']).columns[0]


AttributeError: module 'scripts.domain_logic' has no attribute 'impute_categorical_missing'